In [1]:
import io
import zipfile
import requests
import frontmatter

/Users/nmmsousa/Projects/ai-agents-course/aihero/project/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data


In [3]:
dbt_docs = read_repo_data('dbt-labs', 'docs.getdbt.com')
print(f"dbt documents: {len(dbt_docs)}")

dbt documents: 1388


In [4]:
dbt_docs[0]

{'name': 'converting-notion-to-docusaurus-blog',
 'description': 'Use when converting a Notion markdown export to a Docusaurus blog post for the dbt docs site, or when given a path to a Notion export folder/file.',
 'content': '# Converting Notion Exports to Docusaurus Blog Posts\n\n## Overview\n\nAutomates conversion of Notion markdown exports into properly formatted Docusaurus blog posts for docs.getdbt.com. Handles frontmatter generation, image processing, content cleanup, and file organization.\n\n## When to Use\n\n- User provides a path to a Notion markdown export\n- User mentions converting Notion content to a blog post\n- User has exported content from Notion for the dbt developer blog\n\n## Workflow\n\n### 1. Read Notion Export\n\nParse the markdown file and locate the accompanying image folder (same name as the .md file, without the hash suffix).\n\n### 2. Extract Metadata\n\nFrom the Notion export, extract:\n- **Title**: The H1 heading\n- **Owner**: Maps to author lookup\n- R

In [5]:
for i, doc in enumerate(dbt_docs[:5]):
    content_len = len(doc.get('content', ''))
    title = doc.get('title', 'no title')
    print(f"Doc {i}: {title} — {content_len} chars")

Doc 0: no title — 5157 chars
Doc 1: no title — 4343 chars
Doc 2: no title — 1965 chars
Doc 3: no title — 4343 chars
Doc 4: no title — 10328 chars


In [6]:
# look at all keys available in a document
print(dbt_docs[0].keys())

# look at the full document
print(dbt_docs[0])

dict_keys(['name', 'description', 'content', 'filename'])
{'name': 'converting-notion-to-docusaurus-blog', 'description': 'Use when converting a Notion markdown export to a Docusaurus blog post for the dbt docs site, or when given a path to a Notion export folder/file.', 'content': '# Converting Notion Exports to Docusaurus Blog Posts\n\n## Overview\n\nAutomates conversion of Notion markdown exports into properly formatted Docusaurus blog posts for docs.getdbt.com. Handles frontmatter generation, image processing, content cleanup, and file organization.\n\n## When to Use\n\n- User provides a path to a Notion markdown export\n- User mentions converting Notion content to a blog post\n- User has exported content from Notion for the dbt developer blog\n\n## Workflow\n\n### 1. Read Notion Export\n\nParse the markdown file and locate the accompanying image folder (same name as the .md file, without the hash suffix).\n\n### 2. Extract Metadata\n\nFrom the Notion export, extract:\n- **Title**:

In [7]:
no_title = [doc for doc in dbt_docs if not doc.get('title')]
has_title = [doc for doc in dbt_docs if doc.get('title')]

print(f"Docs with title: {len(has_title)}")
print(f"Docs without title: {len(no_title)}")

Docs with title: 1099
Docs without title: 289


In [8]:
# check what a doc with a title looks like
has_title = [doc for doc in dbt_docs if doc.get('title')]
print(has_title[0])

{'title': 'How to Create Near Real-time Models With Just dbt + SQL', 'description': 'Before I dive into how to create this, I have to say this. **You probably don’t need this**.', 'slug': 'how-to-create-near-real-time-models-with-just-dbt-sql', 'canonical_url': 'https://discourse.getdbt.com/t/how-to-create-near-real-time-models-with-just-dbt-sql/1457', 'authors': ['amy_chen'], 'tags': ['dbt tutorials'], 'hide_table_of_contents': False, 'date': datetime.date(2020, 7, 1), 'is_featured': False, 'content': ':::caution More up-to-date information available\n\nSince this blog post was first published, many data platforms have added support for [materialized views](/blog/announcing-materialized-views), which are a superior way to achieve the goals outlined here. We recommend them over the below approach. \n\n\n:::\n\nBefore I dive into how to create this, I have to say this. **You probably don’t need this**. I, along with my other Fishtown colleagues, have spent countless hours working with c

In [9]:
# check the filenames to understand what we're dealing with
for doc in dbt_docs[:20]:
    print(doc['filename'])

docs.getdbt.com-main/.claude/skills/converting-notion-to-docusaurus-blog/SKILL.md
docs.getdbt.com-main/.claude/skills/docs-issue-triage/SKILL.md
docs.getdbt.com-main/.github/pull_request_template.md
docs.getdbt.com-main/.runlayer/skills/docs-issue-triage/SKILL.md
docs.getdbt.com-main/AGENTS.md
docs.getdbt.com-main/License.md
docs.getdbt.com-main/README.md
docs.getdbt.com-main/contributing/adding-page-components.md
docs.getdbt.com-main/contributing/content-style-guide.md
docs.getdbt.com-main/contributing/content-types.md
docs.getdbt.com-main/contributing/contributor-code-of-conduct.md
docs.getdbt.com-main/contributing/lightbox.md
docs.getdbt.com-main/contributing/operating-model/outline.md
docs.getdbt.com-main/contributing/single-sourcing-content.md
docs.getdbt.com-main/website/blog/2020-07-01-how-to-create-near-real-time-models-with-just-dbt-sql.md
docs.getdbt.com-main/website/blog/2021-02-05-dbt-project-checklist.md
docs.getdbt.com-main/website/blog/2021-02-09-how-to-configure-your-db

In [10]:
# check content length distribution
lengths = [len(doc.get('content', '')) for doc in dbt_docs]
print(f"Average: {sum(lengths)//len(lengths)} chars")
print(f"Max: {max(lengths)} chars")
print(f"Min: {min(lengths)} chars")

Average: 6228 chars
Max: 114796 chars
Min: 0 chars


In [11]:
def is_useful_doc(doc):
    filename = doc.get('filename', '').lower()
    content = doc.get('content', '')
    
    # skip empty files
    if len(content) == 0:
        return False
    
    # skip internal files
    if '.claude' in filename:
        return False
    if '.github' in filename:
        return False
    if '.runlayer' in filename:
        return False
    
    # keep only actual docs and blog posts
    return 'website/docs' in filename or 'website/blog' in filename

dbt_docs_clean = [doc for doc in dbt_docs if is_useful_doc(doc)]
print(f"Original: {len(dbt_docs)}")
print(f"After filtering: {len(dbt_docs_clean)}")

Original: 1388
After filtering: 1163


In [12]:
lengths = [len(doc.get('content', '')) for doc in dbt_docs_clean]

small = [l for l in lengths if l < 2000]
medium = [l for l in lengths if 2000 <= l < 10000]
large = [l for l in lengths if l >= 10000]

print(f"Small docs (< 2000 chars): {len(small)}")
print(f"Medium docs (2000-10000 chars): {len(medium)}")
print(f"Large docs (> 10000 chars): {len(large)}")

Small docs (< 2000 chars): 345
Medium docs (2000-10000 chars): 551
Large docs (> 10000 chars): 267


In [13]:
# check a few docs for headers
for doc in dbt_docs_clean[10:15]:
    content = doc.get('content', '')
    has_headers = '##' in content
    print(f"{doc['filename'].split('/')[-1]} → has headers: {has_headers}")

2021-11-22-primary-keys.md → has headers: True
2021-11-22-sql-surrogate-keys.md → has headers: True
2021-11-23-how-to-upgrade-dbt-versions.md → has headers: True
2021-11-23-so-you-want-to-build-a-package.md → has headers: False
2021-11-26-welcome-to-the-dbt-developer-blog.md → has headers: True


In [15]:
import re

def split_markdown_by_level(text, level=2):
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)
    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        header = parts[i] + parts[i+1]
        header = header.strip()
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()
        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    
    return sections

def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")
    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break
    return result


In [16]:
def chunk_dbt_doc(doc, size=2000, step=1000):
    content = doc.get('content', '')
    doc_copy = doc.copy()
    doc_copy.pop('content')
    
    # small docs — keep as is
    if len(content) < 2000:
        doc_copy['chunk'] = content
        doc_copy['chunk_type'] = 'full'
        return [doc_copy]
    
    # docs with headers — use section splitting
    if '##' in content:
        sections = split_markdown_by_level(content, level=2)
        if len(sections) > 1:
            chunks = []
            for section in sections:
                chunk_doc = doc_copy.copy()
                chunk_doc['chunk'] = section
                chunk_doc['chunk_type'] = 'section'
                chunks.append(chunk_doc)
            return chunks
    
    # large docs without headers — sliding window
    raw_chunks = sliding_window(content, size, step)
    chunks = []
    for raw in raw_chunks:
        chunk_doc = doc_copy.copy()
        chunk_doc['chunk'] = raw['chunk']
        chunk_doc['chunk_type'] = 'sliding_window'
        chunks.append(chunk_doc)
    return chunks


# apply to all docs
dbt_chunks = []
for doc in dbt_docs_clean:
    chunks = chunk_dbt_doc(doc)
    dbt_chunks.extend(chunks)

print(f"Total chunks: {len(dbt_chunks)}")

# see distribution of chunk types
full = [c for c in dbt_chunks if c['chunk_type'] == 'full']
section = [c for c in dbt_chunks if c['chunk_type'] == 'section']
sliding = [c for c in dbt_chunks if c['chunk_type'] == 'sliding_window']

print(f"Full docs: {len(full)}")
print(f"Section chunks: {len(section)}")
print(f"Sliding window chunks: {len(sliding)}")

Total chunks: 4242
Full docs: 345
Section chunks: 3163
Sliding window chunks: 734


In [17]:
# look at one of each type
full_example = [c for c in dbt_chunks if c['chunk_type'] == 'full'][0]
section_example = [c for c in dbt_chunks if c['chunk_type'] == 'section'][0]
sliding_example = [c for c in dbt_chunks if c['chunk_type'] == 'sliding_window'][0]

print("=== FULL DOC ===")
print(f"Length: {len(full_example['chunk'])} chars")
print(full_example['chunk'][:200])

print("\n=== SECTION CHUNK ===")
print(f"Length: {len(section_example['chunk'])} chars")
print(section_example['chunk'][:200])

print("\n=== SLIDING WINDOW ===")
print(f"Length: {len(sliding_example['chunk'])} chars")
print(sliding_example['chunk'][:200])

=== FULL DOC ===
Length: 1236 chars
## What is dbt Mesh?

import Mesh from '/snippets/_what-is-mesh.md';

<Mesh feature={'/snippets/_what-is-mesh.md'} />

<Constant name="dbt" /> is designed to coordinate the features above and simplify

=== SECTION CHUNK ===
Length: 1231 chars
## The right use case

-------------------------------------------

Recently I was working on a JetBlue project and was presented with a legitimate use case: operational data. JetBlue’s Crewmembers ne

=== SLIDING WINDOW ===
Length: 2000 chars
### Why primary keys are important

We all know one of the most fundamental rules in data is that every <Term id="table" /> should have a <Term id="primary-key" />. Primary keys are critical for many 
